# Double rarefaction in PLUTO: Riemann solvers and convergence

The second test of the Riemann solvers, run with the same code and the same workflow as the Sod problem and `nb20`. Two streams of gas move apart and leave a near-vacuum between them:

$$\begin{pmatrix}\rho_l\\ u_l\\ p_l\end{pmatrix} = \begin{pmatrix}1.0\\ -2.0\\ 0.4\end{pmatrix} \qquad\qquad \begin{pmatrix}\rho_r\\ u_r\\ p_r\end{pmatrix} = \begin{pmatrix}1.0\\ +2.0\\ 0.4\end{pmatrix}$$

with $\gamma = 1.4$, a diaphragm at $x = 0.5$ and $t_{\rm stop} = 0.15$. The exact solution is two rarefactions travelling outwards, with $p_{\star} = 1.894\times10^{-3}$ and $\rho_{\star} = 0.0219$: small but not zero. This is the standard test of the robustness of a Riemann solver.

**References**: E. F. Toro, *Riemann Solvers and Numerical Methods for Fluid Dynamics*, 2nd ed., Sect. 4.3.3 (Test 2). M. Zingale, *Introduction to Computational Astrophysical Hydrodynamics*, Eq. 8.109.

## 1. Getting the problem:

The problem directory is in the course repository, packed like the Sod one: [software/test_problems/DoubleRarefaction.zip](https://github.com/wbandabarragan/computational-fluid-dynamics/blob/main/software/test_problems/DoubleRarefaction.zip).

```bash
cd "$HOME/Documents/cfd_simulations"

wget -q --show-progress https://raw.githubusercontent.com/wbandabarragan/computational-fluid-dynamics/main/software/test_problems/DoubleRarefaction.zip

unzip -q DoubleRarefaction.zip

ls -R DoubleRarefaction
```

It holds the same four items as the Sod directory: `init.c` with the new initial condition, `definitions.h`, `pluto.ini` and `other_configs`. Only `init.c` and two numbers in `pluto.ini` differ from the Sod setup.

## 2. The initial condition:

This is the whole of `init.c`. The density and the pressure are the same on both sides, and only the sign of the velocity changes:

```C
void Init (double *v, double x1, double x2, double x3)
{
  #if EOS == IDEAL
   g_gamma = 1.4;
  #endif

  v[RHO] = 1.0;
  v[PRS] = 0.4;

  if (x1 < 0.5) {
    v[VX1] = -2.0;
  }else{
    v[VX1] =  2.0;
  }
  v[VX2] = 0.0;
  v[VX3] = 0.0;
}
```

In `pluto.ini` the only changes with respect to Sod are `tstop 0.15` and the output interval. Everything else, the 512 cells, the CFL number, the outflow boundaries and the `two_shock` solver, is the same.

## 3. Configuring, compiling and running:

Exactly as in Lecture 22, and from inside the new directory. The configuration menu and the build take a couple of minutes; the run itself takes a few seconds.

```bash
cd "$HOME/Documents/cfd_simulations/DoubleRarefaction"

python3 "$PLUTO_DIR/setup.py"
```

then

```bash
make

./pluto
```

With `vtk 0.01` and `tstop 0.15` the run writes 16 snapshots, `data.0000.vtk` to `data.0015.vtk`, plus `vtk.out`, `grid.out` and `pluto.0.log`.

## 4. Reading the output in python:

The same two functions as in Lecture 22, unchanged: `pyvista` opens the VTK file, and the internal energy follows from the pressure and the density.

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
import pyvista as pv
import pandas as pd

In [ ]:
# Path to the simulation and the adiabatic index of the run
sim_dir = "/Users/wbandabarragan/Documents/cfd_simulations/DoubleRarefaction/"

gamma = 1.4

In [ ]:
def read_pluto(number, sim_dir = "./", gamma = 1.4):
    """
    Read a 1D PLUTO VTK file and return x and the primitive variables.
    """

    mesh = pv.read(sim_dir + "data.%04d.vtk" % number)

    # Number of cells and the cell centres along x
    nx = mesh.dimensions[0] - 1

    x_edges = np.linspace(mesh.bounds[0], mesh.bounds[1], nx + 1)
    x = 0.5*(x_edges[:-1] + x_edges[1:])

    # Data arrays, in code units
    rho = pv.get_array(mesh, "rho", preference = 'cell')
    u = pv.get_array(mesh, "vx1", preference = 'cell')
    p = pv.get_array(mesh, "prs", preference = 'cell')

    # Internal energy per unit mass
    e = p/((gamma - 1.)*rho)

    return x, rho, u, p, e

In [ ]:
# Read the simulation times from vtk.out
df_times = pd.read_csv(sim_dir + "vtk.out", sep = r"\s+", header = None)

times = np.array(df_times.iloc[:,1])

# Read the last snapshot
number = 15

x, rho, u, p, e = read_pluto(number, sim_dir, gamma)

print("file %d, t = %.2f, %d cells" % (number, times[number], x.size))

## 5. The four panels:

The same figure as in nb19 and in Lecture 22: density, velocity, pressure and internal energy.

In [ ]:
def plot_pluto(x, rho, u, p, e, t):
    """
    Density, velocity, pressure and internal energy of a 1D PLUTO run.
    """

    fig, ax = plt.subplots(4, 1, figsize = (7,12), sharex = True)

    for axis, var, lab in zip(ax, [rho, u, p, e], [r"$\rho$", r"$u$", r"$p$", r"$e$"]):
        axis.plot(x, var, "o", color = "blue", ms = 3., label = "PLUTO")
        axis.set_ylabel(lab)
        axis.grid(alpha = 0.3)

    ax[0].set_title(r"${\rm Double\ rarefaction,\ }t = %.2f$" % t)
    ax[0].legend(loc = "upper right")
    ax[3].set_xlabel(r"$x$")
    plt.xlim(0., 1.)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_pluto(x, rho, u, p, e, times[number])

## 6. Against the exact solution:

## 7. Instructions for this live exercise:

Repeat the practice tasks of `nb20`.